# Chapter 13: The Training Loop

[Read this chapter online](https://jackluu.io/book/section-4-training/ch13-training-loop/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch13-training-loop.ipynb)

From *Teach Your Computer to Write: Build and Train an LLM from Zero* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 13: The Training Loop

![The map with Training highlighted](https://jackluu.io/book/assets/diagrams/ch13-where-we-are.png)

*Figure 13.1: The model makes predictions, measures its error, and adjusts its weights to improve.*

Our data is batched and ready to go from Chapter 12. We have a model which can predict the next character. Right now, it's no better than random guessing. In this chapter, we will implement the loop that actually trains it on the patterns in the input text.

In this chapter you will:

- Understand gradient descent as walking down a hill.
- See what the learning rate does, by measuring three of them.
- Build the four-step training loop.
- Watch the model learn in real time, and read what its loss means.

> **Words to Know**
>
> - **Optimizer**: The algorithm that updates the model's weights. We use Adam, a popular and steady choice.
> - **Gradient**: The direction we need to move our weights to increase the error. We move in the *opposite* direction to decrease it.
> - **Backpropagation**: The mathematical process of calculating the gradient for every single weight in the model.
> - **Learning Rate**: How far the optimizer moves the weights on each step. If it is too small, training crawls, and if it is too large, it can overshoot.

## Theory

### Walking Down the Hill

How does a model learn? Well, say you are on a bumpy hillside and you're blindfolded. You want to get to the bottom of that hill (where the loss is at its minimum).

You would not be able to see how far down that hill goes but you could feel the direction of the hill around you. If the hill was higher to your right you would know to step to your left.

![Gradient descent as walking down a hill](https://jackluu.io/book/assets/diagrams/ch13-gradient-descent.png)

*Figure 13.2: The optimizer takes small steps down the loss landscape to lower the loss.*

This is **gradient descent** (Figure 13.2). The slope under your feet is the gradient, calculated by backpropagation, and taking a small step downhill is the optimizer updating the weights.

### The Four-Step Loop

Training is a repetitive cycle that we run thousands of times (Figure 13.3):

![The training loop cycle](https://jackluu.io/book/assets/diagrams/ch13-training-cycle.png)

*Figure 13.3: The four steps of the training loop.*

1.  **Forward Pass**: We pass a batch of data through the model to get its predictions.
2.  **Calculate Loss**: We compare the predictions to the correct targets using cross-entropy.
3.  **Backward Pass**: PyTorch automatically works out the gradient for every weight in the model (`loss.backward()`). The part of PyTorch that does this is called autograd.
4.  **Optimizer Step**: The optimizer adjusts the weights slightly in the right direction (`optimizer.step()`).

### How Big Should the Step Be?

Gradient descent allows you to know what direction to descend the mountain. It does not allow you to know how far to walk down the mountain. This is known as your **learning rate**. It is something most newbies will get wrong.

Our config sets it to 0.0003. To see where that comes from, train the same model three times from the same starting weights, changing only the learning rate:

```python
for lr in (3e-3, 3e-4, 3e-5):
    torch.manual_seed(42)                       # same starting weights every time
    model = GPT(GPTConfig())
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
```

```text
$ python src/examples/ch13_learning_rate.py
lr=0.003    start 4.33  after 150 steps 2.35
lr=0.0003   start 4.33  after 150 steps 2.60
lr=3e-05    start 4.33  after 150 steps 3.31
```

Most people need the lesson on the bottom row. At 0.00003 we are taking such small steps that we have hardly moved by 150 steps. The loss is at 3.31 compared to 4.17 if we guessed randomly. So it's learning just extremely slowly.

You may wonder about the top row. With 10x the learning rate we got there *faster* with a loss of 2.35 versus our learning rate which reached 2.60. Why doesn't the book use this?

Because 150 steps is not 3,000 steps. When taking larger steps you run the risk of bouncing off the bottom of the valley or blowing up. Our example only ran for 150 steps so we don't know. 0.0003 will get us to 1.74 with the whole example. You could try running 0.003 for 3000 steps and see what happens.

### The "Aha!" Moment: It Learned

At the beginning of training, loss is about 4.3. Recall from Chapter 11 that completely random guessing between 65 characters has an expected loss of 4.17. Initially our model is even worse than random!

But as the loop runs, the numbers start to move.

![Training and validation loss falling over 3,000 steps](https://jackluu.io/book/assets/diagrams/ch13-loss-chart.png)

*Figure 13.4: Over 3,000 steps, the training loss falls from 4.33 to 1.74.*

The loss falls quickly (Figure 13.4). By step 300 it is already at 2.7, and by step 3,000 it reaches 1.74.

### What the Loss Number Actually Means

It's pretty difficult to understand what a loss of 1.74 means by itself. So we can convert this to a probability.

As you can see there is only one value used for cross-entropy. It's the probability the model assigned to the actual next character. Cross-entropy takes this value and uses its negative logarithm as the loss. Undoing the logarithm allows us to get the probability that we can understand:

```python
for name, loss in [("random guessing", math.log(VOCAB)),
                   ("step 1", 4.3280),
                   ("step 300", 2.7016),
                   ("step 3000", 1.7357)]:
    # Cross-entropy is the negative log of the probability given to the right answer
    prob = math.exp(-loss)
    print(f"{name:<16} loss {loss:.2f} -> {prob:6.1%} on the right character")
```

```text
$ python src/examples/ch13_loss_meaning.py
random guessing  loss 4.17 ->   1.5% on the right character
step 1           loss 4.33 ->   1.3% on the right character
step 300         loss 2.70 ->   6.7% on the right character
step 3000        loss 1.74 ->  17.6% on the right character
```

At the start, the model assigns around 1.3% probability to the correct character. This is slightly worse than the 1.5% we'd get if we picked from the 65 possible characters at random. At the end of training it has increased this probability 12x which is great. Though we still have our model guessing the wrong answer 4 out of every 5 times.

Keep this in mind when thinking about your model. It knows what letters follow other letters. It knows about word length, end of line and who is speaking. It doesn't know English. And it definitely doesn't know what any of this means. We will show you this in Chapter 17. Our model will write "Praviour soul to shall that that are the and not,". You can see Shakespeare but no meaning.

The "Training" section of our map is finished. Our model knows how to generate the next character in its corpus.

## Code

The core loop is simple to write in PyTorch.

```python
optimizer = torch.optim.Adam(model.parameters(), lr=train_cfg.learning_rate)

for step in range(1, train_cfg.max_iters + 1):
    x, y, train_iter = get_batch(train_loader, train_iter, device)

    # Forward pass
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, gpt_cfg.vocab_size), y.view(-1))

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
```

Run the full script to watch the loss fall:

```text
$ python src/ch12_train.py
Chapter 13: The Training Loop

Model parameters: 824,832
Training on     : cpu
Steps           : 3,000
Batch size      : 32
Block size      : 128

Starting training... (eval every 300 steps)
step     1/3000 | train loss: 4.3280 | val loss: 4.2114 | elapsed: 3s | ETA: 7595s
step   300/3000 | train loss: 2.7016 | val loss: 2.5433 | elapsed: 55s | ETA: 496s
...
step   900/3000 | train loss: 2.2633 | val loss: 2.1889 | elapsed: 160s | ETA: 372s
step  1200/3000 | train loss: 2.1185 | val loss: 2.1053 | elapsed: 210s | ETA: 314s
step  1500/3000 | train loss: 2.0174 | val loss: 2.0078 | elapsed: 248s | ETA: 248s
```

**What just happened:**

1.  Line 1 creates an Adam optimizer to handle the weight updates.
2.  Line 3 indicates 3,000 steps of training have been completed. The log above reached halfway through after 248 seconds, so that entire run took roughly eight minutes on whatever machine produced it. On the author's 2020 laptop (see Preface), that same run took 20-30 minutes.
3.  Lines 11-13 compute gradients and perform the weight update. During the run you see the training loss drop from 4.33 to 1.74. Two losses are printed to the log. The train loss is calculated on text that the model sees as it trains. The val loss is calculated on the held-out text from Chapter 4.
4.  We saved the trained weights to a file so we can load them later.

**Shape Check:**

Table 13.1 lists the parameter count.

**Table 13.1:** Total adjustable parameters in the language model.

| Variable | Shape | Meaning |
| :--- | :--- | :--- |
| `model.parameters()` | `824,832` | The total number of individual numbers (weights) the optimizer is adjusting. |

## Try It

We can see the mechanics of PyTorch's automatic gradients on a tiny scale.

```python
"""Show how PyTorch automatically calculates gradients to update a weight."""
import torch

# A single weight starting at 2.0
weight = torch.tensor([2.0], requires_grad=True)

# Our simple 'model' multiplies input by weight
x = torch.tensor([3.0])
target = torch.tensor([12.0]) # We want output to be 12

print(f"Initial weight: {weight.item():.2f}")

for step in range(3):
    # Forward pass
    output = weight * x
    loss = (output - target) ** 2

    # Backward pass (calculate the gradient)
    loss.backward()

    print(f"Step {step+1}: Output={output.item():.2f}, "
          f"Loss={loss.item():.2f}, Gradient={weight.grad.item():.2f}")

    # Update weight (move in opposite direction of gradient)
    with torch.no_grad():
        weight -= 0.05 * weight.grad
        weight.grad.zero_()

print(f"Final weight: {weight.item():.2f}")
```

Lines 19, 26, and 27 show PyTorch computing the gradient and using it to adjust the weight toward the target.

```text
$ python src/examples/ch13_autograd_demo.py
Initial weight: 2.00
Step 1: Output=6.00, Loss=36.00, Gradient=-36.00
Step 2: Output=11.40, Loss=0.36, Gradient=-3.60
Step 3: Output=11.94, Loss=0.00, Gradient=-0.36
Final weight: 4.00
```

> **Try It**
>
> Open `src/examples/ch13_autograd_demo.py`. Change the starting `weight` to `10.0` and watch how the gradient pulls it down instead of pushing it up, always aiming for the target output of 12.
>

> **In Business**
>
> That's where you spend money. When you train your assistant with your house style, your business is spending money on compute time to iterate this process billions of times on thousands of documents. You will want to monitor your loss curve as a key metric. As long as validation loss is decreasing, your assistant is getting better at mimicking your style.
>

> **Watch Out**
>
> Always remember to run `optimizer.zero_grad()` before `loss.backward()`. By default, PyTorch accumulates gradients (it sums them). If you forget to zero them before starting the backward pass, your model will take a step based on a mix of the current batch and all the previous batches and will thus go in the wrong direction.

## Key Takeaways

- Training is a loop of four steps: Forward, Loss, Backward, Step.
- Gradient descent finds the lowest loss by taking small steps downhill.
- We use the `torch.optim.Adam` optimizer to manage the complex math of updating the weights.
- The model starts by guessing blindly (loss > 4.17), but over 3,000 steps, it learns the patterns of the text and the loss falls.

## Check Your Understanding

1. What are the four main steps inside the training loop?
2. What does `loss.backward()` actually do?
3. Why is a dropping loss curve a good sign?

## Assignment

> **Assignment 13: Shrink the step**
>
> **About 15 minutes. You change one number and write no new code.** Work in this chapter's Colab notebook or on your own laptop.
>
> 1. **Run it.** Run `src/examples/ch13_autograd_demo.py` as it is. Copy the five lines it prints: the initial weight, the three steps and the final weight.
> 2. **Change one number.** Find the line `weight -= 0.05 * weight.grad`. The `0.05` is the learning rate: how far the weight moves on each step. Change `0.05` to `0.0003` and run the script again.
> 3. **Compare.** Which printed lines are the same in both runs? After three steps, how close does the output get to the target of 12 in each run? Is the loss still falling in the second run?
> 4. **Explain it.** In two or three sentences a manager could follow, say what a learning rate that is too small for the job costs a company that pays for every training step.
>
> **Hand in:** both sets of five lines, your answers to step 3, and your sentences.

## Further Reading

**How a network learns anything at all.** If a network with multiple layers outputs the wrong answer, there would be no way of knowing which middle weight to blame. This paper allowed us to take the error and push it back through the network, blaming each weight in proportion to how much it moved the answer. All models in this book are trained using backpropagation. All models that are in use today train this way. In our training loop, `loss.backward()` handles this.

**The optimizer on one line of your training loop.** Backpropagation tells you which direction each weight should move, but doesn't determine how far. Adam gives each weight its own step size, tailored based on how rapidly that weight has been changing recently. Rarely used weights can take bigger steps. Weights which change erratically take smaller steps to calm down. Adam is our default optimizer in the training loop in Chapter 13 and most loops elsewhere.


Kingma, D. P., & Ba, J. (2014). *Adam: A method for stochastic optimization* (arXiv:1412.6980). arXiv. https://doi.org/10.48550/arXiv.1412.6980

Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). Learning representations by back-propagating errors. *Nature, 323*, 533–536. https://doi.org/10.1038/323533a0

---

### `src/ch12_train.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch12_train.py"   # a cell has none, and the file uses it to find the text

"""
Implement the main training loop.
This file belongs to Chapter 13.
Run: python src/ch12_train.py
"""
import os
import time
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import sys

from src.utils.config import GPTConfig, TrainConfig
from src.ch09_gpt_model import GPT
from src.ch11_dataloader import train_dataset, val_dataset

# Settings
gpt_cfg   = GPTConfig()
train_cfg = TrainConfig()

# --- The Idea ---

# Disable gradient tracking for evaluation to save memory and compute
@torch.no_grad()
def estimate_val_loss(model, val_loader, device):
    # Set model to evaluation mode (disables dropout)
    model.eval()
    val_iter = iter(val_loader)
    losses = []
    for _ in range(min(50, len(val_loader))):
        x, y = next(val_iter)
        x, y = x.to(device), y.to(device)
        logits = model(x)
        # Compute average loss for this batch
        loss = F.cross_entropy(logits.view(-1, gpt_cfg.vocab_size), y.view(-1))
        losses.append(loss.item())

    # Return to training mode
    model.train()
    return sum(losses) / len(losses)

def get_batch(loader, loader_iter, device):
    # Fetch the next batch, restarting the iterator if needed
    try:
        x, y = next(loader_iter)
    except StopIteration:
        loader_iter = iter(loader)
        x, y = next(loader_iter)
    return x.to(device), y.to(device), loader_iter

# --- Demo ---
if __name__ == "__main__":
    print("Chapter 13: The Training Loop\n")

    torch.manual_seed(42)
    device = "cpu"

    model = GPT(gpt_cfg).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}")
    print(f"Training on     : {device}")
    print(f"Steps           : {train_cfg.max_iters:,}")
    print(f"Batch size      : {train_cfg.batch_size}")
    print(f"Block size      : {gpt_cfg.block_size}")

    optimizer = torch.optim.Adam(model.parameters(), lr=train_cfg.learning_rate)

    train_loader = DataLoader(
        train_dataset, batch_size=train_cfg.batch_size, shuffle=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=train_cfg.batch_size, shuffle=False
    )
    train_iter = iter(train_loader)

    print(f"\nStarting training... (eval every {train_cfg.eval_interval} steps)")
    start_time = time.time()
    train_losses = []

    for step in range(1, train_cfg.max_iters + 1):
        x, y, train_iter = get_batch(train_loader, train_iter, device)

        # Forward pass
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, gpt_cfg.vocab_size), y.view(-1))

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

        if step % train_cfg.eval_interval == 0 or step == 1:
            val_loss = estimate_val_loss(model, val_loader, device)
            rec_losses = train_losses[-train_cfg.eval_interval:]
            avg_train = sum(rec_losses) / len(rec_losses)
            elapsed = time.time() - start_time
            steps_left = train_cfg.max_iters - step
            eta = (elapsed / step) * steps_left if step > 0 else 0

            print(f"step {step:5d}/{train_cfg.max_iters} | "
                  f"train loss: {avg_train:.4f} | "
                  f"val loss: {val_loss:.4f} | "
                  f"elapsed: {elapsed:.0f}s | "
                  f"ETA: {eta:.0f}s")

    os.makedirs(train_cfg.checkpoint_dir, exist_ok=True)
    checkpoint_path = os.path.join(train_cfg.checkpoint_dir, "model.pt")

    torch.save({
        "model_state": model.state_dict(),
        "gpt_cfg"    : gpt_cfg,
        "step"       : train_cfg.max_iters,
        "val_loss"   : val_loss,
    }, checkpoint_path)

    total_time = time.time() - start_time
    print(f"\nTraining complete! Total time: {total_time:.0f}s")
    print(f"Checkpoint saved to: {checkpoint_path}")
    print("Ready for Chapter 14 (checkpointing) and Chapter 15 (generation)!")

---

### `src/examples/ch13_autograd_demo.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch13_autograd_demo.py"   # a cell has none, and the file uses it to find the text

"""Show how PyTorch automatically calculates gradients to update a weight."""
import torch

# A single weight starting at 2.0
weight = torch.tensor([2.0], requires_grad=True)

# Our simple 'model' multiplies input by weight
x = torch.tensor([3.0])
target = torch.tensor([12.0]) # We want output to be 12

print(f"Initial weight: {weight.item():.2f}")

for step in range(3):
    # Forward pass
    output = weight * x
    loss = (output - target) ** 2
    
    # Backward pass (calculate the gradient)
    loss.backward()
    
    print(f"Step {step+1}: Output={output.item():.2f}, "
          f"Loss={loss.item():.2f}, Gradient={weight.grad.item():.2f}")
    
    # Update weight (move in opposite direction of gradient)
    with torch.no_grad():
        weight -= 0.05 * weight.grad
        weight.grad.zero_()

print(f"Final weight: {weight.item():.2f}")

---

### `src/examples/ch13_learning_rate.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch13_learning_rate.py"   # a cell has none, and the file uses it to find the text

"""
Train the same model three times with three learning rates, and compare.
This file belongs to Chapter 13.
Run: python src/examples/ch13_learning_rate.py
"""
import os
import sys

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from src.ch09_gpt_model import GPT
from src.ch11_dataloader import train_dataset
from src.utils.config import GPTConfig, TrainConfig

STEPS = 150

for lr in (3e-3, 3e-4, 3e-5):
    torch.manual_seed(42)                       # same starting weights every time
    model = GPT(GPTConfig())
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = iter(DataLoader(train_dataset, batch_size=TrainConfig().batch_size,
                             shuffle=True))

    first = last = None
    for step in range(STEPS):
        x, y = next(loader)
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, GPTConfig().vocab_size), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if first is None:
            first = loss.item()
        last = loss.item()

    print(f"lr={lr:<8g} start {first:.2f}  after {STEPS} steps {last:.2f}")

---

### `src/examples/ch13_loss_meaning.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch13_loss_meaning.py"   # a cell has none, and the file uses it to find the text

"""
Translate a cross-entropy loss into the confidence it stands for.
This file belongs to Chapter 13.
Run: python src/examples/ch13_loss_meaning.py
"""
import math

VOCAB = 65

# Losses from the training run in this chapter
for name, loss in [("random guessing", math.log(VOCAB)),
                   ("step 1", 4.3280),
                   ("step 300", 2.7016),
                   ("step 3000", 1.7357)]:
    # Cross-entropy is the negative log of the probability given to the right answer
    prob = math.exp(-loss)
    print(f"{name:<16} loss {loss:.2f} -> {prob:6.1%} on the right character")